In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# Automatically use Kaggle's GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

file_path = '/kaggle/input/datasets/soumilnegi154/shakespeare/shakespeare.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
char2int = {ch: i for i, ch in enumerate(chars)}
int2char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

print(f"Total characters: {len(text)}, Unique characters: {vocab_size}")

encoded_text = [char2int[ch] for ch in text]
seq_length = 100 
X_data = []
y_data = []

for i in range(0, len(encoded_text) - seq_length):
    X_data.append(encoded_text[i : i + seq_length])
    y_data.append(encoded_text[i + 1 : i + seq_length + 1]) 

X = torch.tensor(X_data, dtype=torch.long)
y = torch.tensor(y_data, dtype=torch.long)

batch_size = 128
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

class ShakespeareLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(ShakespeareLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden):
        embedded = self.embedding(x) 
        out, hidden = self.lstm(embedded, hidden)
        out = out.reshape(-1, self.hidden_size)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        hidden = (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                  torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))
        return hidden

embedding_dim = 64
hidden_size = 256
num_layers = 2
epochs = 10
learning_rate = 0.001

model = ShakespeareLSTM(vocab_size, embedding_dim, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Starting Training...")
for epoch in range(epochs):
    model.train()
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        current_batch_size = inputs.size(0)
        hidden = model.init_hidden(current_batch_size)
        
        optimizer.zero_grad()
        outputs, hidden = model(inputs, hidden)
        loss = criterion(outputs, targets.view(-1))
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        
        if batch_idx % 200 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(dataloader)}] | Loss: {loss.item():.4f}")

def generate_text(model, start_str, length, temperature=0.8):
    model.eval()
    hidden = model.init_hidden(1)
    
    input_seq = torch.tensor([char2int[ch] for ch in start_str], dtype=torch.long).unsqueeze(0).to(device)
    
    for i in range(len(start_str) - 1):
        _, hidden = model(input_seq[:, i].unsqueeze(1), hidden)
        
    current_char = input_seq[:, -1].unsqueeze(1)
    generated_text = start_str
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(current_char, hidden)
            output = output / temperature
            probs = torch.softmax(output, dim=1).squeeze()
            char_idx = torch.multinomial(probs, 1).item()
            generated_text += int2char[char_idx]
            current_char = torch.tensor([[char_idx]], dtype=torch.long).to(device)
            
    return generated_text

print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.8))

In [ ]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.7))

In [ ]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.1))

In [ ]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.5))

In [ ]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=0.9))

In [ ]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=1))

In [ ]:
print(generate_text(model, start_str="O Romeo, Romeo! wherefore art thou ", length=500, temperature=1))